# 03 EEA-Batch-Verarbeitung

## Zweck

Verwenden Sie die historischen Luftqualitätsdaten EEA als erforderliche Datei-/Batch-Quelle. Normalisieren Sie lokale EEA-Zeilen, ordnen Sie Stationen `city_id` zu, führen Sie Datenqualitätsprüfungen durch, aggregieren Sie auf Stadt-/Tag-/Schadstoffebene und schreiben Sie Silver Parquet.

## Eingaben

- `data/silver/city_reference.parquet` aus Phase 2.
- Lokale EEA CSV oder Parquet-Dateien unter `data/bronze/eea/`.
– Wenn keine lokale EEA-Datei vorhanden ist, erstellt das Notebook eine kleine kontrollierte Stichprobe unter `data/samples/`, um die Transformation reproduzierbar zu halten.

## Ausgaben

- `data/silver/eea_city_daily.parquet`

## Verwendete Technologien

Python, Pandas, pyarrow, Parquet, Jupyter Notebook.

## Konfiguration

Hier wird kein EEA-Download durchgeführt. Generierte Beispiel-/Ausgabedaten werden von Git ignoriert. Historische EEA-Daten werden nicht mit Open-Meteo API-Daten gemischt.

In [ ]:
from pathlib import Path
import os
import json
import pandas as pd

_env_root = os.getenv("PROJECT_ROOT")
if _env_root:
    PROJECT_ROOT = Path(_env_root).resolve()
elif Path.cwd().name == "notebooks":
    PROJECT_ROOT = Path.cwd().parent
else:
    PROJECT_ROOT = Path.cwd()

DATA_DIR = PROJECT_ROOT / Path(os.getenv("DATA_DIR", "data"))
CHECKPOINT_DIR = PROJECT_ROOT / Path(os.getenv("CHECKPOINT_DIR", "data/checkpoints"))
CITY_REFERENCE_PATH = DATA_DIR / "silver" / "city_reference.parquet"

## Umsetzung

Die Verarbeitung lädt Dateidaten mit pandas, bereinigt und normalisiert Spalten, schreibt Parquet-Dateien und validiert das erneute Einlesen. Der Umfang ist auf PM2.5, PM10 und NO2 begrenzt.


### Schadstoff- und Stationszuordnungen konfigurieren

Nur PM2.5, PM10 und NO2 sind im Geltungsbereich. Die Zuordnungstabelle verbindet ausgewählte EEA-Stationskennungen mit den stabilen `city_id`-Werten des Projekts.

In [ ]:
from datetime import datetime, timedelta, timezone
from pathlib import Path
import pandas as pd

BRONZE_EEA_DIR = DATA_DIR / "bronze" / "eea"
SAMPLES_DIR = DATA_DIR / "samples"
SILVER_DIR = DATA_DIR / "silver"
for path in [BRONZE_EEA_DIR, SAMPLES_DIR, SILVER_DIR]:
    path.mkdir(parents=True, exist_ok=True)

CORE_POLLUTANTS = {"pm2_5", "pm10", "no2"}
POLLUTANT_LABEL_MAP = {
    "PM2.5": "pm2_5",
    "PM2,5": "pm2_5",
    "Particles < 2.5 µm (aerodynamic diameter)": "pm2_5",
    "PM10": "pm10",
    "Particles < 10 µm (aerodynamic diameter)": "pm10",
    "NO2": "no2",
    "Nitrogen dioxide": "no2",
    "Nitrogen dioxide (air)": "no2",
}

STATION_MAPPING = pd.DataFrame([
    {"city_id": "vienna_at", "eea_station_id": "AT90TAB", "mapping_status": "selected"},
    {"city_id": "berlin_de", "eea_station_id": "DEBE068", "mapping_status": "selected"},
    {"city_id": "paris_fr", "eea_station_id": "FR04143", "mapping_status": "selected"},
    {"city_id": "madrid_es", "eea_station_id": "ES0118A", "mapping_status": "selected"},
    {"city_id": "rome_it", "eea_station_id": "IT1906A", "mapping_status": "selected"},
    {"city_id": "amsterdam_nl", "eea_station_id": "NL00014", "mapping_status": "selected"},
    {"city_id": "warsaw_pl", "eea_station_id": "PL0592A", "mapping_status": "selected"},
    {"city_id": "prague_cz", "eea_station_id": "CZ0ARIE", "mapping_status": "selected"},
])

city_reference_df = pd.read_parquet(CITY_REFERENCE_PATH)
assert set(STATION_MAPPING["city_id"]).issubset(set(city_reference_df["city_id"])), \
    f"STATION_MAPPING-city_ids fehlen in der Städtereferenz: {set(STATION_MAPPING['city_id']) - set(city_reference_df['city_id'])}"


### Definieren Sie ein kontrolliertes EEA-Fallback-Beispiel

Wenn kein lokaler EEA-Extrakt vorhanden ist, erstellt dieser Helfer deterministische Beispielmessungen. Sie beweisen nur die Mechanik und werden mit `data_status=controlled_sample_fallback` beibehalten.

In [ ]:
def create_controlled_eea_sample(path: Path) -> Path:
    base_date = datetime(2023, 1, 1, tzinfo=timezone.utc)
    rows = []
    for day_offset in range(30):
        date = base_date + timedelta(days=day_offset)
        for hour in range(0, 24, 8):
            ts = (date + timedelta(hours=hour)).strftime("%Y-%m-%dT%H:%M:%SZ")
            for station_index, station in enumerate(STATION_MAPPING["eea_station_id"]):
                variation = (day_offset % 7) + (hour // 8)
                rows.extend([
                    {"AirQualityStationEoICode": station, "DatetimeBegin": ts, "AirPollutant": "PM2.5", "Concentration": round(8.0 + variation + station_index * 0.5, 1), "Unit": "µg/m³"},
                    {"AirQualityStationEoICode": station, "DatetimeBegin": ts, "AirPollutant": "PM10", "Concentration": round(18.0 + variation + station_index * 0.8, 1), "Unit": "µg/m³"},
                    {"AirQualityStationEoICode": station, "DatetimeBegin": ts, "AirPollutant": "Nitrogen dioxide", "Concentration": round(30.0 + variation + station_index * 1.2, 1), "Unit": "µg/m³"},
                ])
    pd.DataFrame(rows).to_csv(path, index=False)
    return path


### Suchen Sie nach kompatiblen Quellspalten

Echte EEA-Exporte können unterschiedliche Spaltennamen verwenden. Dieser kleine Helfer wählt den ersten unterstützten Kandidaten aus und schlägt eindeutig fehl, wenn keiner vorhanden ist.

In [ ]:
def first_existing(df: pd.DataFrame, candidates: list[str]) -> str:
    for candidate in candidates:
        if candidate in df.columns:
            return candidate
    raise KeyError(f"Erwartete Spalte fehlt. Geprüft wurden: {candidates}")


### Rohe EEA-Zeilen normalisieren

Der Loader liest CSV oder Parquet, standardisiert Spaltennamen und -typen, entfernt fehlerhafte Zeilen, schließt negative Werte aus und begrenzt Schadstoffe auf den Projektumfang.

In [ ]:
def load_eea_raw(path: Path) -> pd.DataFrame:
    raw = pd.read_parquet(path) if path.suffix.lower() == ".parquet" else pd.read_csv(path)
    total_rows = len(raw)
    station_col = first_existing(raw, ["AirQualityStationEoICode", "AirQualityStation", "station_id"])
    ts_col = first_existing(raw, ["DatetimeBegin", "datetime_begin", "timestamp", "date"])
    pollutant_col = first_existing(raw, ["AirPollutant", "pollutant", "Pollutant", "Component"])
    value_col = first_existing(raw, ["Concentration", "concentration", "Value", "value"])
    unit_col = first_existing(raw, ["Unit", "unit"])
    df = pd.DataFrame({
        "eea_station_id": raw[station_col].astype(str).str.strip(),
        "datetime_begin": pd.to_datetime(raw[ts_col], utc=True, errors="coerce"),
        "pollutant": raw[pollutant_col].astype(str).str.strip().map(POLLUTANT_LABEL_MAP),
        "value": pd.to_numeric(raw[value_col], errors="coerce"),
        "unit": raw[unit_col].astype(str).str.strip(),
    })
    after_parse = len(df)
    df = df.dropna(subset=["eea_station_id", "datetime_begin", "pollutant", "value", "unit"])
    after_dropna = len(df)
    df = df[df["value"] >= 0].copy()
    after_negative = len(df)
    df = df[df["pollutant"].isin(CORE_POLLUTANTS)].copy()
    after_pollutant = len(df)
    print(
        f"load_eea_raw: {total_rows} Rohzeilen → "
        f"{after_parse - after_dropna} verworfen (Nullwerte/nicht zugeordnet) → "
        f"{after_dropna - after_negative} verworfen (negative Werte) → "
        f"{after_negative - after_pollutant} verworfen (Schadstoffe außerhalb des Umfangs) → "
        f"{after_pollutant} beibehalten"
    )
    return df.reset_index(drop=True)


### Stationen zu Städten zuordnen

Die Stationszuordnung führt die explizite Brücke von EEA-Stationskennungen zum gemeinsamen Stadtmodell durch.

In [ ]:
def map_stations_to_cities(raw_df: pd.DataFrame) -> pd.DataFrame:
    selected = STATION_MAPPING.query("mapping_status == 'selected'")[["eea_station_id", "city_id"]]
    return raw_df.merge(selected, on="eea_station_id", how="inner")


### Aggregieren Sie Beobachtungen auf Stadttagsebene

Die abschließende Transformation berechnet den täglichen Mittelwert, das Minimum, das Maximum und die Beobachtungsanzahl für jede Stadt und jeden Schadstoff.

In [ ]:
def aggregate_to_city_daily(mapped_df: pd.DataFrame) -> pd.DataFrame:
    required = ["city_id", "datetime_begin", "pollutant", "value", "unit"]
    missing = [col for col in required if col not in mapped_df.columns]
    if missing:
        raise ValueError(f"Erforderliche Spalten fehlen: {missing}")
    df = mapped_df.copy()
    df["datetime_begin"] = pd.to_datetime(df["datetime_begin"], utc=True, errors="coerce")
    if df[required].isna().any().any():
        raise ValueError("Erforderliche EEA-Felder enthalten Nullwerte")
    df = df[df["value"].ge(0) & df["pollutant"].isin(CORE_POLLUTANTS)].copy()
    df["date"] = df["datetime_begin"].dt.date
    daily = df.groupby(["city_id", "date", "pollutant", "unit"], as_index=False).agg(
        mean_value=("value", "mean"),
        min_value=("value", "min"),
        max_value=("value", "max"),
        observation_count=("value", "count"),
    )
    daily["source"] = "eea"
    daily["processing_time_utc"] = pd.Timestamp(datetime.now(timezone.utc))
    return daily


### Wählen Sie den Eingang EEA aus

Das Notebook bevorzugt einen echten CSV oder Parkettauszug unter `data/bronze/eea/`. Wenn keine vorhanden ist, wird die kontrollierte Probe generiert und diese Herkunft erfasst.

In [ ]:
source_files = sorted(BRONZE_EEA_DIR.glob("*.csv")) + sorted(BRONZE_EEA_DIR.glob("*.parquet"))
if source_files:
    eea_input_path = source_files[0]
    eea_data_status = "real_eea_file"
    print(f"Reale EEA-Datei wird verwendet: {eea_input_path}")
else:
    eea_input_path = create_controlled_eea_sample(SAMPLES_DIR / "eea_controlled_sample.csv")
    eea_data_status = "controlled_sample_fallback"
    print(f"Keine EEA-Datei gefunden in {BRONZE_EEA_DIR} — kontrollierte Beispieldaten werden verwendet: {eea_input_path}")


### Laden Sie die ausgewählte Datei

Parsing-Fehler werden mit einer umsetzbaren Meldung umschlossen, damit Schüler inkompatible EEA-Extrakte diagnostizieren können.

In [ ]:
try:
    raw_eea = load_eea_raw(eea_input_path)
except Exception as exc:
    raise RuntimeError(
        f"EEA-Datei konnte nicht geladen werden '{eea_input_path}': {exc}. "
        "Prüfe, ob die Datei eine gültige CSV- oder Parquet-Datei ist und dem erwarteten Spaltenschema entspricht."
    ) from exc


### Wenden Sie die Stationszuordnung an

Diese Aktion zeigt, wie viele normalisierte Zeilen ausgewählten Städten zugewiesen werden konnten.

In [ ]:
mapped_eea = map_stations_to_cities(raw_eea)
print(f"Stationszuordnung: {len(raw_eea)} Zeilen → {len(mapped_eea)} Zeilen wurden city_ids zugeordnet")


### Schreiben Sie den EEA Silver-Datensatz

Die tägliche Aggregation erhält ihre Herkunftsmarkierung und wird als Parquet für die Goldschicht gespeichert.

In [ ]:
eea_city_daily = aggregate_to_city_daily(mapped_eea)
eea_city_daily["data_status"] = eea_data_status
output_path = SILVER_DIR / "eea_city_daily.parquet"
eea_city_daily.to_parquet(output_path, index=False)
eea_city_daily.head()


## Validierung / Qualitätsprüfungen

Validieren Sie den Schadstoffumfang, das erforderliche Schema, die Zuordnungsabdeckung, nicht-negative Werte, die Aggregationsanzahl und das Parquet-Rücklesen.

In [ ]:
required_output_columns = {
    "city_id", "date", "pollutant", "mean_value", "min_value", "max_value",
    "observation_count", "unit", "source", "processing_time_utc", "data_status",
}
missing_cols = required_output_columns - set(eea_city_daily.columns)
assert not missing_cols, f"In der Ausgabe fehlen erforderliche Spalten: {missing_cols}"

unexpected_pollutants = set(eea_city_daily["pollutant"]) - CORE_POLLUTANTS
assert not unexpected_pollutants, f"Die Ausgabe enthält unerwartete Schadstoffe: {unexpected_pollutants}"

assert (eea_city_daily["observation_count"] >= 1).all(), \
    f"Gefunden: {(eea_city_daily['observation_count'] < 1).sum()} Zeilen mit observation_count gleich null"
assert (eea_city_daily["source"] == "eea").all(), \
    f"Unerwartete source-Werte: {eea_city_daily['source'].unique()}"
assert set(eea_city_daily["data_status"]).issubset({"real_eea_file", "controlled_sample_fallback"}), \
    f"Unerwartete data_status-Werte: {eea_city_daily['data_status'].unique()}"

roundtrip = pd.read_parquet(output_path)
assert len(roundtrip) == len(eea_city_daily), \
    f"Abweichende Zeilenanzahl beim Parquet-Roundtrip: geschrieben: {len(eea_city_daily)}, erneut gelesen: {len(roundtrip)}"

print("Zusammenfassung nach Schadstoff:")
roundtrip.groupby("pollutant")["observation_count"].sum()

## Ergebnisse

Phase 3 erzeugt `eea_city_daily.parquet` als historischen Silver-Luftqualitätsdatensatz. Wenn kein echter EEA-Extrakt vorhanden ist, basiert die Ausgabe auf einer klar gekennzeichneten kontrollierten Probe und muss für die endgültige Analyse durch echte EEA-Daten ersetzt werden.

## Einschränkungen

Die Zuordnung von Station zu Stadt ist vereinfacht und muss anhand echter EEA-Stationsmetadaten überprüft werden. Die kontrollierte Probe ist nur ein Ersatz für die Reproduzierbarkeit, kein analytischer Beweis.

## Nächster Schritt

Führen Sie das Notebook `04_wikipedia_web_scraping.ipynb` aus, um kontextbezogene Stadtmetadaten aus Wikipedia zu erstellen.